1. Считавание датасета из файла train.csv и получение общей информации о содержимом

In [ ]:
import pandas as pd
df = pd.read_csv('train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


2) информация о датасете

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


3. Подсчет процента выживаемости у каждого класса пассажиров (Pclass)

In [16]:
percent_survival = (
    df.groupby('Pclass')['Survived']
      .mean()
      .mul(100)
      .round(2)
      .rename('Survival, percent')
)
print(percent_survival)

Pclass
1    62.96
2    47.28
3    24.24
Name: Survival, percent, dtype: float64


4. и 5. Самое популярное мужское и самое популярное женское имя на корабле и в каждом классе

In [31]:
import pandas as pd
import re

df = pd.read_csv('train.csv')

PARENTHESES = re.compile(r"\(([^)]*)\)")
QUOTED = re.compile(r'"[^"]*"')


def extract_first_name(full_name: str) -> str:
    rest = full_name.split(".", 1)[1] if "." in full_name else full_name

    match = PARENTHESES.search(rest)
    if match:
        inside = match.group(1).strip()
        if not inside.startswith('"'):
            rest = inside

    rest = QUOTED.sub("", rest)
    rest = PARENTHESES.sub("", rest)

    words = rest.split()
    return words[0] if words else ""


def first_names(frame) -> pd.Series:
    return frame["Name"].apply(extract_first_name)


def most_popular(frame, sex: str) -> tuple[str, int]:
    counts = first_names(frame[frame["Sex"] == sex]).value_counts()
    if counts.empty:
        return "—", 0
    return counts.index[0], int(counts.iloc[0])


def run(frame) -> None:
    print("*** Самые популярные имена на корабле ***")
    for sex, label in (("male", "мужское"), ("female", "женское")):
        name, count = most_popular(frame, sex)
        print(f"{label}: {name} ({count} чел.)")


def most_popular_by_class(frame):
    rows = []
    for pclass in sorted(frame["Pclass"].unique()):
        in_class = frame[frame["Pclass"] == pclass]
        for sex, label in (("male", "мужское"), ("female", "женское")):
            name, count = most_popular(in_class, sex)
            rows.append({"Pclass": pclass, "пол": label, "имя": name, "человек": count})
    return pd.DataFrame(rows).set_index(["Pclass", "пол"])


def run_by_class(frame) -> None:
    print("*** Самые популярные имена в каждом классе ***")
    print(most_popular_by_class(frame))


if __name__ == "__main__":
    run(df)
    run_by_class(df)

*** Самые популярные имена на корабле ***
мужское: William (35 чел.)
женское: Anna (15 чел.)
*** Самые популярные имена в каждом классе ***
                      имя  человек
Pclass пол                        
1      мужское    William       11
       женское  Elizabeth        5
2      мужское    William        9
       женское  Elizabeth        5
3      мужское    William       15
       женское       Anna        9


6. и 7. Вывод части таблицы с пассажирами, возраст которых больше 44 лет, а также с пассажирами мужского пола, возраст которых меньше 44 лет

In [40]:
def older_44(df, age: float = 44):
    return df[df["Age"] > age]


def run(df):
    print("*** Часть таблицы с пассажирами, возраст которых больше 44 лет ***")
    print(older_44(df, 44))


if __name__ == "__main__":
    run(df)

def under_44(df, age: float = 44):

    return df[(df["Age"] > 0) & (df["Age"] < age) & (df["Sex"] == "male")]


def run_under_44(df):
    print("*** Часть таблицы с пассажирами мужского пола, возраст которых меньше 44 лет ***")
    print(under_44(df, 44))

if __name__ == "__main__":
    run_under_44(df)

*** Часть таблицы с пассажирами, возраст которых больше 44 лет ***
     PassengerId  Survived  Pclass  \
6              7         0       1   
11            12         1       1   
15            16         1       2   
33            34         0       2   
52            53         1       1   
..           ...       ...     ...   
857          858         1       1   
862          863         1       1   
871          872         1       1   
873          874         0       3   
879          880         1       1   

                                                  Name     Sex   Age  SibSp  \
6                              McCarthy, Mr. Timothy J    male  54.0      0   
11                            Bonnell, Miss. Elizabeth  female  58.0      0   
15                    Hewlett, Mrs. (Mary D Kingcome)   female  55.0      0   
33                               Wheadon, Mr. Edward H    male  66.0      0   
52            Harper, Mrs. Henry Sleeper (Myna Haxtun)  female  49.0      1   
..

8. Количество n-местных кабин (в которых было 2, 3, 4, ... человека)

In [46]:
def people_in_the_cabin(df):

    return df["Cabin"].dropna().value_counts()


def cabin_distribution(df, min_people: int = 2):

    sizes = people_in_the_cabin(df).value_counts().sort_index()
    sizes.index.name = "человек в каюте"
    sizes.name = "количество кают"
    return sizes[sizes.index >= min_people]


def run(df) -> None:
    print("*** Количество n-местных кают ***")
    print(cabin_distribution(df, min_people=2))

if __name__ == "__main__":
  run(df)

*** Количество n-местных кают ***
человек в каюте
2    38
3     5
4     3
Name: количество кают, dtype: int64
